# Class 2: Prompt Engineering Basics


### Table of Contents

| # | Section | Time | Type |
|---|---------|------|------|
| 1 | Intro -- Why Two People Get Wildly Different Results from the Same AI | 10 min | Lecture + Live Demo |
| 2 | Concept in 5 -- The Anatomy of a Great Prompt (RCTFC) | 5 min | Analogy + Framework |
| 3 | Tool Demo -- Zero-Shot vs Few-Shot Prompting | 15 min | Live Side-by-Side |
| 4 | Hands-On -- Chain-of-Thought and Output Formatting | 25 min | Guided Colab Cells |
| 5 | Guided Exercise -- Refining a Prompt in Five Rounds | 20 min | Iterative Live Rewrite |
| 6 | Mini Build -- Three Prompt Challenges | 25 min | Applied Exercises |
| 7 | Wrap-Up -- The Prompt Engineer's Checklist | 10 min | Recap + Framework |



---

**Important:** Every section in this class builds on a single continuous scenario. Read the scenario setup carefully -- everything that follows depends on it.

---
---

# THE SCENARIO -- Sam's System Design Review

Sam is a backend developer who just joined a product team at an e-commerce company. Yesterday, his tech lead ran a 50-minute system design review for a new checkout service. The lead took rough notes on a whiteboard, snapped a photo, and dropped the transcription into Slack with the message: *"Sam, can you turn this into something the team can actually use? Need it by EOD."*

Here are the raw notes, exactly as posted:

---

> **System Design Review -- Checkout Service Rewrite**
> **11 April 2025, 10:00-10:50 AM**
>
> Attendees: Maya (Tech Lead), Sam (Backend Dev), Rina (Frontend Dev), Jatin (SRE), Pooja (Product Manager)
>
> - current checkout monolith hitting 4s response times under peak load, target is sub-500ms
> - agreed to split into 3 microservices: cart-service, payment-service, order-service
> - payment-service must be PCI compliant, jatin said we need dedicated VPC
> - sam to prototype cart-service in Go by 25 apr, maya reviews
> - rina needs new API contracts for cart endpoints by 20 apr so frontend work can start
> - database: postgres for orders, redis for cart state -- jatin wants connection pooling from day one
> - event-driven arch for order completion: payment-service publishes event, order-service consumes
> - message broker: debated kafka vs rabbitmq, no decision, maya wants a comparison doc by 18 apr, nobody assigned
> - pooja raised concern: what happens to in-flight carts during deployment? need blue-green or canary strategy
> - monitoring: jatin will set up datadog dashboards, needs service names and key metrics list by 22 apr
> - load testing: jatin wants to run a load test before go-live, no date set, depends on cart-service prototype
> - API rate limiting: discussed but no decision on strategy or thresholds
> - rollback plan: not discussed, maya said add it to next review agenda
> - next review: 18 apr same time

---

We will use these exact notes throughout the entire class. Every prompting technique will be demonstrated by helping Sam get progressively better results from the same raw material.

---
---

# SECTION 1 -- INTRO: WHY TWO PEOPLE GET WILDLY DIFFERENT RESULTS FROM THE SAME AI

**Time: 10 minutes** | **Goal:** Establish that the quality gap between useful and useless AI output is almost always the prompt, not the model.

> "Last class, we set up accounts on three AI platforms and compared their personalities. We learned about tokens, context windows, and temperature. Good foundation.
>
> But here's the thing nobody told you: the gap between a useless AI response and a brilliant one almost never comes down to which model you're using. It comes down to how you ask.
>
> Two developers on the same team, using the same model, will get dramatically different outputs based on how they write their prompts. One gets a generic wall of text. The other gets something they can paste directly into a Jira ticket.
>
> That skill -- the skill of writing prompts that produce exactly what you need -- has a name: **prompt engineering**. And it's what we're spending the next two hours on.
>
> Let me show you the difference. Same notes, same model, two completely different prompts."

### Live Demo: Same Notes, Two Prompts

*[Run both prompts below in the same AI platform, one after the other. Let the class see both outputs side by side.]*

**Prompt A -- What Sam types on his first attempt:**

In [ ]:
# PROMPT A -- No structure, no guidance
# Copy the text below (without # marks) into ChatGPT, Claude, or AI Studio

prompt_a = """
Summarise this meeting.

System Design Review -- Checkout Service Rewrite
11 April 2025, 10:00-10:50 AM
Attendees: Maya (Tech Lead), Sam (Backend Dev), Rina (Frontend Dev), Jatin (SRE), Pooja (Product Manager)

- current checkout monolith hitting 4s response times under peak load, target is sub-500ms
- agreed to split into 3 microservices: cart-service, payment-service, order-service
- payment-service must be PCI compliant, jatin said we need dedicated VPC
- sam to prototype cart-service in Go by 25 apr, maya reviews
- rina needs new API contracts for cart endpoints by 20 apr so frontend work can start
- database: postgres for orders, redis for cart state -- jatin wants connection pooling from day one
- event-driven arch for order completion: payment-service publishes event, order-service consumes
- message broker: debated kafka vs rabbitmq, no decision, maya wants a comparison doc by 18 apr, nobody assigned
- pooja raised concern: what happens to in-flight carts during deployment? need blue-green or canary strategy
- monitoring: jatin will set up datadog dashboards, needs service names and key metrics list by 22 apr
- load testing: jatin wants to run a load test before go-live, no date set, depends on cart-service prototype
- API rate limiting: discussed but no decision on strategy or thresholds
- rollback plan: not discussed, maya said add it to next review agenda
- next review: 18 apr same time
"""

print("Paste the prompt text into your AI platform.")
print("Read the output carefully. Is it something Sam could actually send to his team?")

Paste the prompt text into your AI platform.
Read the output carefully. Is it something Sam could actually send to his team?


**What the AI typically returns:** a flat paragraph restating the notes. Something like: *"The team met to discuss rewriting the checkout service. They decided to split it into three microservices. Various tasks were assigned..."*



## Output of prompt_a →

### System Design Review — Checkout Service Rewrite (11 Apr 2025)

**Goal**

* Current checkout monolith at ~4s under peak; target **<500 ms**.
* Plan to rewrite using microservices.

**Architecture Decisions**

* Split into **cart-service**, **payment-service**, **order-service**
* **Event-driven flow**: payment-service publishes → order-service consumes
* **Database**:

  * Postgres for orders
  * Redis for cart state
  * Connection pooling required from day one
* **Security**: payment-service must be **PCI compliant** with a dedicated VPC

**Open Technical Decisions**

* Message broker: Kafka vs RabbitMQ (comparison doc requested)
* API rate limiting: strategy and thresholds undecided
* Deployment handling for in-flight carts (blue-green vs canary)
* Rollback plan (deferred to next review)

**Action Items**

* **Sam**: Prototype cart-service in Go by **25 Apr** (Maya to review)
* **Rina**: Needs cart API contracts by **20 Apr**
* **Maya**: Requested Kafka vs RabbitMQ comparison doc by **18 Apr** (no owner assigned)
* **Jatin**:

  * Set up Datadog dashboards
  * Needs service names + key metrics by **22 Apr**
  * Run load test before go-live (after prototype ready)

**Risks / Concerns**

* In-flight carts during deployment
* No message broker decision yet
* No rate limiting plan
* No rollback strategy
* Load testing date not defined

**Next Meeting**

* **18 Apr**, same time


It reads like a middling summary -- technically accurate, practically useless. No clear separation between decisions and open questions. No dependency tracking. No ownership table. Sam's tech lead would look at this and say: *"I could have just re-read my own notes."*

Now watch what happens with a structured prompt:

In [ ]:
# PROMPT B -- Structured with Role, Context, Task, Format, Constraints
# Copy this into the SAME platform and compare

prompt_b = """
Role: You are a senior software architect who writes clear,
actionable design review summaries for engineering teams.

Context: I'm a backend developer on a product team. We just held
a 50-minute system design review for rewriting our checkout service
from a monolith to microservices. The summary will be posted in our
engineering Slack channel and referenced during sprint planning,
so it must be precise and scannable.

Task: Turn the raw meeting notes below into a structured
design review summary.

Format:
1. Overview (what was reviewed, key goal -- max 2 sentences)
2. Architecture Decisions (numbered list of what was agreed)
3. Action Items (table: Action | Owner | Deadline | Blocked By)
4. Open Questions (bulleted list of unresolved items)
5. Risks Identified (bulleted list)

Constraints:
- Keep the total summary under 500 words.
- If the notes do not specify an owner or deadline, write "TBD".
- Do not invent details not present in the notes.
- Use precise technical language but avoid unnecessary jargon.

Notes:
[Paste Sam's design review notes here]
"""

print("Paste the prompt text into your AI platform.")
print("Compare this output to the first one. Same model. Same notes.")

Paste the prompt text into your AI platform.
Compare this output to the first one. Same model. Same notes.


**What the AI returns now:** a cleanly sectioned summary with architecture decisions separated from action items, a table tracking owners and blockers, and distinct lists for open questions and risks. Sam can paste this directly into Slack or a Confluence page.

## Output of prompt_b →

#### 1. Overview

The team reviewed the redesign of the checkout system to replace the current monolith (≈4s peak latency) with a microservices architecture targeting sub-500ms response times. The discussion covered service boundaries, data stores, event flow, infrastructure requirements, and delivery milestones.

#### 2. Architecture Decisions

1. Checkout will be split into three services: **cart-service**, **payment-service**, and **order-service**.
2. **Event-driven order completion**: payment-service publishes an event; order-service consumes it.
3. **Data storage**:

   * Postgres for order data
   * Redis for cart state
   * Connection pooling required from initial implementation
4. **Payment-service** must be **PCI compliant** and deployed in a **dedicated VPC**.
5. Monitoring will use **Datadog dashboards** for service-level visibility.

#### 3. Action Items

| Action                                               | Owner              | Deadline | Blocked By                    |
| ---------------------------------------------------- | ------------------ | -------- | ----------------------------- |
| Prototype cart-service in Go; review prototype       | Sam (review: Maya) | 25 Apr   | None                          |
| Provide cart-service API contracts for frontend      | TBD                | 20 Apr   | None                          |
| Produce Kafka vs RabbitMQ comparison doc             | TBD                | 18 Apr   | None                          |
| Provide service names and key metrics for monitoring | TBD                | 22 Apr   | None                          |
| Set up Datadog dashboards                            | Jatin              | TBD      | Service names and key metrics |
| Run load test before go-live                         | Jatin              | TBD      | Cart-service prototype        |
| Define deployment strategy for in-flight carts       | TBD                | TBD      | None                          |
| Add rollback plan to next review agenda              | Maya               | 18 Apr   | None                          |

#### 4. Open Questions

* Kafka vs RabbitMQ for message broker
* Deployment strategy for in-flight carts (blue-green vs canary)
* API rate limiting strategy and thresholds
* Rollback strategy for new architecture
* Load testing schedule and success criteria
* Ownership for API contract definition
* Ownership for metrics/service name definition

#### 5. Risks Identified

* Current performance gap (4s → <500ms) may require significant optimization beyond service split
* No message broker selected; blocks event-driven implementation details
* Undefined handling of in-flight carts during deployments may cause user-facing failures
* No rate limiting strategy; risk of service overload under peak traffic
* Rollback plan not defined; increases deployment risk
* Load testing not scheduled; performance target may not be validated before go-live


**The model didn't get smarter between the two prompts.** It was equally capable both times. The structured prompt simply gave it the information it needed to do its job -- the same way a well-written Jira ticket gets resolved faster than one that just says "it's broken."

That structure has a name, and it's what we're learning today.

> "So what exactly made Prompt B better? It followed a five-part structure that works for every prompt you'll ever write. Let me break it down."

---
---

# SECTION 2 -- CONCEPT IN 5: THE ANATOMY OF A GREAT PROMPT (RCTFC)

**Time: 5 minutes** | **Goal:** Give students a repeatable five-part framework they can apply to any prompt, starting today.

> "Here's the analogy that will stick with you:
>
> **Prompting an AI is like filing a really good bug report.**
>
> We've all received bug reports that just say 'the app is broken.' Useless. You don't know what page they were on, what they were trying to do, what they expected, or what happened. You have to guess.
>
> Now compare that to a report with steps to reproduce, environment details, expected vs actual behavior, and a screenshot. Same developer reading it. Completely different outcome.
>
> AI responds to specificity the same way you respond to a good bug report. The more context up front, the less guessing, and the better the result.
>
> The framework is called **RCTFC**. Five parts. Works everywhere."

### The RCTFC Framework

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/189/471/original/2.png?1776085355)


**Mapping Prompt B to RCTFC:**

| Element | What Prompt B Said | What It Prevented |
|---------|-------------------|-------------------|
| **Role** | "senior software architect" | Prevented intern-level surface summary |
| **Context** | "backend dev, design review, posted in Slack for sprint planning" | Prevented wrong audience/tone |
| **Task** | "Turn raw notes into structured summary" | Prevented rambling or wrong deliverable |
| **Format** | "Overview, Decisions, Action Items table, Open Questions, Risks" | Prevented wall-of-text output |
| **Constraints** | "Under 500 words, TBD for unknowns, don't invent" | Prevented hallucination and bloat |

**The bug report parallel:**

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/189/469/original/1.png?1776085157)


Both follow the same principle: **specificity in, quality out.**

---

**Beginner Note:** You don't need all five elements every time. For a simple question like "What does a 502 error mean?", just the Task is enough. But for anything requiring more than a one-line answer, RCTFC is your go-to.

**Advanced Note:** In production systems, Role + Context + Constraints are often encoded as a persistent system prompt, while Task + Format change per request. We'll cover system prompts later.

> "RCTFC gives you the structure. But sometimes the AI's output is close but doesn't quite match the specific format your team uses. That's where the next technique comes in: instead of describing the format, you SHOW it."

---
---

# SECTION 3 -- TOOL DEMO: ZERO-SHOT VS FEW-SHOT PROMPTING

**Time: 15 minutes** | **Goal:** Students understand when and why to include examples in prompts.

> "Prompt B was a **zero-shot** prompt -- instructions but zero examples. For most tasks, that's enough.
>
> But sometimes you need output to match a very specific format -- your company's ADR template, your team's Jira ticket style, your org's incident report structure. Instructions alone can't capture those nuances.
>
> **The analogy: onboarding a contractor to your codebase.** Telling a contractor 'follow our coding standards' is zero-shot -- they guess. Opening three of your best pull requests and saying 'write code like this' is few-shot. Same contractor, dramatically better first PR.
>
> Let me show you this with Sam's notes."

In [ ]:
# PROMPT -- Zero-shot: no examples

prompt_zero = """
Extract the action items from the design review notes below
and format them as a structured list.

Notes:
[Paste Sam's design review notes here]
"""

print("Zero-shot: The AI WHAT to do but not WHAT the output should look like.")
print("It will return a reasonable list -- but in its own default format, not your team's.")

Zero-shot: The AI WHAT to do but not WHAT the output should look like.
It will return a reasonable list -- but in its own default format, not your team's.


## Output of prompt_zero →

#### Action Items

1. **Prototype cart-service in Go**

   * Owner: Sam
   * Deadline: 25 Apr
   * Notes: Maya to review prototype

2. **Provide cart-service API contracts for frontend**

   * Owner: TBD
   * Deadline: 20 Apr
   * Notes: Required for frontend work to begin

3. **Create Kafka vs RabbitMQ comparison document**

   * Owner: TBD
   * Deadline: 18 Apr
   * Notes: Requested by Maya

4. **Provide service names and key metrics for monitoring**

   * Owner: TBD
   * Deadline: 22 Apr
   * Notes: Required for Datadog dashboards

5. **Set up Datadog dashboards**

   * Owner: Jatin
   * Deadline: TBD
   * Notes: Depends on service names and metrics list

6. **Run load test before go-live**

   * Owner: Jatin
   * Deadline: TBD
   * Notes: Depends on cart-service prototype

7. **Define deployment strategy for in-flight carts (blue-green or canary)**

   * Owner: TBD
   * Deadline: TBD
   * Notes: Raised by Pooja

8. **Add rollback plan discussion to next review agenda**

   * Owner: Maya
   * Deadline: 18 Apr
   * Notes: To be addressed in next design review

**Typical zero-shot output:** a simple bulleted list.

Correct information, but not in Sam's team format. No priority, no dependency tracking, no status, no decision reference.

Now with three examples:

In [ ]:
# PROMPT -- Few-shot: three examples of the desired format

prompt_few = """
Extract the action items from the design review notes below.
Format each action item EXACTLY like the examples shown.

=== EXAMPLES OF OUR TEAM FORMAT ===

TASK: Migrate user-auth service to OAuth 2.0
OWNER: Deepa (Backend)
DEADLINE: 3 Mar 2025
PRIORITY: P1 - Critical
STATUS: In Progress
BLOCKED BY: None
DECISION REF: Auth strategy agreed in review #12

TASK: Write integration tests for payment webhook handler
OWNER: Karthik (QA)
DEADLINE: 7 Mar 2025
PRIORITY: P2 - High
STATUS: Not Started
BLOCKED BY: Payment service API contract (Deepa, due 1 Mar)
DECISION REF: Webhook pattern agreed in review #11

TASK: Evaluate CloudWatch vs Datadog for new services
OWNER: TBD
DEADLINE: 10 Mar 2025
PRIORITY: P2 - High
STATUS: Not Started
BLOCKED BY: None
DECISION REF: Monitoring discussion in review #12 (no decision)

=== NOW EXTRACT FROM THESE NOTES ===

Notes:
[Paste Sam's design review notes here]
"""

print("Few-shot: Same task, but the AI can SEE the exact format to replicate.")
print("Notice the new fields: PRIORITY, BLOCKED BY, DECISION REF.")

Few-shot: Same task, but the AI can SEE the exact format to replicate.
Notice the new fields: PRIORITY, BLOCKED BY, DECISION REF.


## Output of prompt_few →

1. >TASK: Prototype cart-service in Go
>
>OWNER: Sam (Backend)
>
>DEADLINE: 25 Apr 2025
>
>PRIORITY: P1 - Critical
>
>STATUS: Not Started
>
>BLOCKED BY: None
>
>DECISION REF: Microservices split agreed in checkout rewrite review (11 Apr 2025)

2. >TASK: Provide cart-service API contracts for frontend
>
>OWNER: TBD
>
>DEADLINE: 20 Apr 2025
>
>PRIORITY: P1 - Critical
>
>STATUS: Not Started
>
>BLOCKED BY: None
>
>DECISION REF: cart-service introduction agreed in checkout rewrite review (11 Apr 2025)

3. >TASK: Create Kafka vs RabbitMQ comparison document
>
>OWNER: TBD
>
>DEADLINE: 18 Apr 2025
>
>PRIORITY: P2 - High
>
>STATUS: Not Started
>
>BLOCKED BY: None
>
>DECISION REF: Message broker discussion in checkout rewrite review (11 Apr 2025)



**What changes:** the model now produces action items in the exact seven-field structure. It assigns priorities based on context. It fills in BLOCKED BY relationships (load testing blocked by cart-service prototype). It writes "TBD" for the unassigned Kafka comparison doc.

### When to Use Which

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/189/477/original/3.png?1776086243)


**Real-world parallel:** Teams don't describe their conventions in abstract prose -- they point to examples. "Write tests like this one." "Structure your endpoints like that one." The examples ARE the instruction.

> "RCTFC gives you structure. Few-shot gives you format control. But what about tasks where the AI needs to actually reason -- not just organize text, but think through a problem? That's the next technique."

---
---

# SECTION 4 -- HANDS-ON: CHAIN-OF-THOUGHT AND OUTPUT FORMATTING

**Time: 25 minutes** | **Goal:** Students learn chain-of-thought for analytical tasks and precise output format control.

> "Here's a technique that will change how you use AI for analysis.
>
> **The analogy: rubber duck debugging.** Every experienced developer knows this -- when you're stuck, explaining the problem step by step to a rubber duck (or a colleague) often reveals the answer. The act of articulating your reasoning catches errors you'd miss by jumping to a conclusion.
>
> LLMs work the same way. When you tell the model to think through a problem step by step before answering, the reasoning itself improves.
>
> The technique is called **chain-of-thought prompting**, and it's one instruction: 'Think through this step by step before giving your final answer.'
>
> Sam's tech lead asks: 'What are the three biggest risks to shipping on time?' Let's see the difference."

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/189/481/original/4.png?1776086966)

In [ ]:
# PROMPT -- Direct question, no chain-of-thought

prompt_no_cot = """
Based on these design review notes, what are the three biggest
risks that could delay the checkout service rewrite?

Notes:
[Paste Sam's design review notes here]
"""

print("No chain-of-thought: the model jumps straight to listing risks.")
print("Does it catch DEPENDENCY CHAINS or just list obvious items?")

No chain-of-thought: the model jumps straight to listing risks.
Does it catch DEPENDENCY CHAINS or just list obvious items?


**Typical output without chain-of-thought:** three plausible items:
1. *Unresolved Message Broker Decision*
2. *Frontend Blocked on Missing API Contracts*
3. *Deployment Strategy for In-Flight Carts Not Defined*

## Output of prompt_no_cot →

#### 1. Unresolved Message Broker Decision (Kafka vs RabbitMQ)

* The architecture depends on an event-driven flow where `payment-service` publishes and `order-service` consumes.
* The broker choice is still undecided, and Maya requested a comparison document with **no owner assigned**.
* This blocks:

  * Service integration contracts
  * Infrastructure provisioning
  * Event schema decisions
  * Load testing setup
* Because this is a foundational dependency, any delay cascades across multiple services.



#### 2. Frontend Blocked on Missing API Contracts

* Rina needs **cart API contracts by April 20**, but they are not yet defined.
* Sam is scheduled to **prototype cart-service by April 25**, which is after the frontend start date.
* This creates a sequencing conflict:

  * Frontend work cannot start
  * Integration testing slips
  * Downstream UI validation delayed
* This cross-team dependency risks stalling parallel development.



#### 3. Deployment Strategy for In-Flight Carts Not Defined

* Pooja raised concern about **in-flight carts during deployment**.
* Blue-green or canary strategy discussed, but **no decision or owner assigned**.
* This impacts:

  * Rollout safety
  * Data consistency for Redis cart state
  * Go-live readiness
  * Load testing realism
* Release approval could be blocked late without this resolved.


and the output continues......

Not wrong, but shallow. It misses that the API contract (due Apr 20) blocks all frontend work. It misses that the Kafka vs RabbitMQ comparison has no owner, blocking the event architecture decision, blocking both payment-service and order-service. It misses that the rollback plan wasn't even discussed.

Now with chain-of-thought:

In [ ]:
# PROMPT -- Chain-of-thought: structured reasoning before the answer

prompt_cot = """
Based on these design review notes, what are the three biggest
risks that could delay the checkout service rewrite?

Before answering, work through this analysis step by step:

Step 1 -- Map every task and its deadline onto a timeline.
Step 2 -- Identify which tasks depend on other tasks completing first.
          Draw out the dependency chains.
Step 3 -- Flag any task that has no owner, no deadline, or an
          unresolved decision blocking it.
Step 4 -- Identify which delays would cascade and affect the most
          downstream work.
Step 5 -- Consider what was NOT discussed but should have been.

Then, based on that analysis, name the three highest-impact risks
and explain the specific dependency chain that makes each one dangerous.

Notes:
[Paste Sam's design review notes here]
"""

print("Chain-of-thought: the model reasons through dependencies before answering.")
print("Compare the depth of analysis to the previous version.")

Chain-of-thought: the model reasons through dependencies before answering.
Compare the depth of analysis to the previous version.


**What changes:** the model maps the full dependency graph. It catches that the unassigned Kafka vs RabbitMQ doc blocks the event architecture, which blocks both payment-service and order-service. It catches that the API contract (Apr 20) gates all frontend work. It flags the missing rollback plan as a launch blocker.


## Output of prompt_cot →

#### Checkout Service Rewrite — Risk Analysis

###### Step 1 — Timeline of Tasks & Deadlines

| Date   | Task                                       | Owner                        | Notes                              |
| ------ | ------------------------------------------ | ---------------------------- | ---------------------------------- |
| Apr 18 | Message broker comparison doc              | ❌ Unassigned                 | Kafka vs RabbitMQ decision blocker |
| Apr 20 | Cart API contracts ready                   | Rina (needs), backend TBD    | Frontend blocked without this      |
| Apr 22 | Service names + metrics for Datadog        | Jatin (needs), team provides | Depends on service definitions     |
| Apr 25 | Cart-service Go prototype                  | Sam                          | Maya reviews                       |
| TBD    | Load testing                               | Jatin                        | Depends on cart-service prototype  |
| TBD    | Deployment strategy (blue/green or canary) | ❌ Unassigned                 | In-flight cart handling            |
| TBD    | API rate limiting strategy                 | ❌ Unassigned                 | Architecture decision              |
| TBD    | Rollback plan                              | ❌ Unassigned                 | Release safety                     |
| TBD    | PCI-compliant VPC for payment-service      | Jatin                        | Infra prerequisite                 |

---

#### Step 2 — Dependency Chains

##### Chain A — Event Architecture

```
Message broker decision
        ↓
Event schema definition
        ↓
payment-service publish logic
        ↓
order-service consumer
        ↓
end-to-end checkout flow
        ↓
load testing
        ↓
go-live
```

---

##### Chain B — Cart Service → Frontend

```
Cart API contracts
        ↓
Frontend implementation
        ↓
Integration testing
        ↓
Checkout UX validation
        ↓
Release readiness
```

But also:

```
Cart-service prototype (Apr 25)
        ↓
Final API validation
        ↓
Frontend integration
```

This creates a **timeline conflict**.

---

##### Chain C — Release Safety / Deployment

```
Deployment strategy (blue/green/canary)
        ↓
Handling in-flight carts
        ↓
Rollback design
        ↓
Load testing realism
        ↓
Go-live approval
```

---

##### Chain D — Observability

```
Service definitions
        ↓
Metrics list
        ↓
Datadog dashboards (Apr 22)
        ↓
Load testing visibility
        ↓
Performance validation
```

---

#### Step 3 — Tasks Missing Owner / Decision / Deadline

High-risk gaps:

* Message broker decision — **no owner**
* Deployment strategy — **no owner**
* Rollback plan — **not discussed**
* API rate limiting — **no decision**
* Load testing — **no date**
* PCI VPC timeline — **no deadline**
* API contracts — **deadline exists but no assigned backend owner**

and the output continues.....

**Key insight:** chain-of-thought didn't just change the format -- it changed the **quality of the thinking**.

### When to Use Chain-of-Thought

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/189/487/original/5.png?1776087247)

**Rule of thumb:** If a good answer requires more than one mental step, add chain-of-thought. If you'd whiteboard it before answering, tell the AI to whiteboard it too.

**Real-world parallel:** This is how senior engineers run architecture reviews -- they map dependencies, identify unknowns, trace failure modes, then recommend. Chain-of-thought forces the AI to follow the same discipline.

> "Great analysis means nothing if the reader has to reformat it.
>
> **The analogy: an API response contract.** When you build an API, you define a response schema so consumers know what to expect. Prompting works the same way: define the output schema, get predictable results.
>
> Sam needs to share the review results with four different audiences. Same information, four formats."

In [ ]:
# FORMAT A -- Slack update (numbered list)

prompt_slack = """
Summarise the key decisions from this design review as a numbered
list for posting in our engineering Slack channel.

Rules:
- Each item is exactly one sentence, in past tense ("The team agreed to...")
- Maximum 7 items
- Technical but accessible to the full engineering org

Notes:
[Paste Sam's notes here]
"""

print("Format A: Scannable Slack update. One sentence per decision.")

Format A: Scannable Slack update. One sentence per decision.


## output of prompt_slack →

1. The team agreed to decompose the checkout monolith into three microservices (cart-service, payment-service, and order-service) to meet the sub-500ms performance target.
2. The team agreed that the payment-service must be PCI compliant and run within a dedicated VPC.
3. The team agreed to use PostgreSQL for order data and Redis for cart state, with connection pooling implemented from the start.
4. The team agreed to adopt an event-driven architecture where the payment-service publishes order completion events consumed by the order-service.
5. The team agreed that Sam would prototype the cart-service in Go by 25 April with a review by Maya, and Rina would receive API contracts by 20 April to begin frontend work.
6. The team agreed that Jatin would set up Datadog dashboards, with required service names and key metrics to be provided by 22 April.
7. The team agreed to defer decisions on the message broker, API rate limiting strategy, deployment handling of in-flight carts, load testing schedule, and rollback planning to the next review.


In [ ]:
# FORMAT B -- Sprint planning table

prompt_table = """
Extract all action items from the design review notes and present
them as a table with these exact columns:

| Task | Owner | Deadline | Priority | Blocked By |
|------|-------|----------|----------|------------|

Rules:
- Blocked By = the specific task that must complete first.
  Write "None" if independent.
- If owner or deadline is missing, write "TBD".
- Priority: P0 (launch blocker), P1 (critical path), P2 (important).

Notes:
[Paste Sam's notes here]
"""

print("Format B: Ready for sprint planning or Jira import.")

Format B: Ready for sprint planning or Jira import.


## output of prompt_jira →

| Task                                                                           | Owner | Deadline    | Priority | Blocked By                                                |
| ------------------------------------------------------------------------------ | ----- | ----------- | -------- | --------------------------------------------------------- |
| Prototype cart-service in Go and submit for review                             | Sam   | 25 Apr 2025 | P1       | None                                                      |
| Review cart-service prototype                                                  | Maya  | TBD         | P1       | Prototype cart-service in Go and submit for review        |
| Provide API contracts for cart-service endpoints                               | TBD   | 20 Apr 2025 | P1       | None                                                      |
| Produce Kafka vs RabbitMQ comparison document                                  | TBD   | 18 Apr 2025 | P1       | None                                                      |
| Design deployment strategy for handling in-flight carts (blue-green or canary) | TBD   | TBD         | P0       | None                                                      |
| Set up dedicated VPC for PCI-compliant payment-service                         | Jatin | TBD         | P0       | None                                                      |
| Implement connection pooling for Postgres and Redis                            | Jatin | TBD         | P1       | None                                                      |
| Provide service names and key metrics list for monitoring                      | TBD   | 22 Apr 2025 | P1       | None                                                      |
| Set up Datadog dashboards                                                      | Jatin | TBD         | P1       | Provide service names and key metrics list for monitoring |
| Plan and execute load testing before go-live                                   | Jatin | TBD         | P0       | Prototype cart-service in Go and submit for review        |
| Define API rate limiting strategy and thresholds                               | TBD   | TBD         | P1       | None                                                      |
| Add rollback planning to next design review agenda                             | Maya  | 18 Apr 2025 | P2       | None                                                      |


In [ ]:
# FORMAT C -- JSON for automation

prompt_json = """
Return the action items from these design review notes as a JSON array.
Each object must have exactly these keys:

[
  {
    "task": "Prototype cart-service in Go",
    "owner": "Sam (Backend)",
    "deadline": "2025-04-25",
    "priority": "P1",
    "status": "not_started",
    "blocked_by": null,
    "service": "cart-service"
  }
]

Rules:
- deadline in YYYY-MM-DD format. null if not specified.
- priority: P0, P1, or P2
- service: cart-service, payment-service, order-service, or infrastructure
- Return ONLY valid JSON. No explanation before or after.

Notes:
[Paste Sam's notes here]
"""

print("Format C: Machine-readable JSON for Jira API, dashboards, or CI/CD.")

Format C: Machine-readable JSON for Jira API, dashboards, or CI/CD.


## output of prompt_json →



```
[
  {
    "task": "Prototype cart-service in Go",
    "owner": "Sam (Backend)",
    "deadline": "2025-04-25",
    "priority": "P1",
    "status": "not_started",
    "blocked_by": null,
    "service": "cart-service"
  },
  {
    "task": "Review cart-service prototype",
    "owner": "Maya (Tech Lead)",
    "deadline": null,
    "priority": "P1",
    "status": "not_started",
    "blocked_by": "Prototype cart-service in Go",
    "service": "cart-service"
  },
  {
    "task": "Provide API contracts for cart-service endpoints",
    "owner": "TBD",
    "deadline": "2025-04-20",
    "priority": "P1",
    "status": "not_started",
    "blocked_by": null,
    "service": "cart-service"
  },
  {
    "task": "Produce Kafka vs RabbitMQ comparison document",
    "owner": "TBD",
    "deadline": "2025-04-18",
    "priority": "P1",
    "status": "not_started",
    "blocked_by": null,
    "service": "infrastructure"
  },
........
```
and the output continues


In [ ]:
# FORMAT D -- Architecture Decision Record

prompt_adr = """
Draft an Architecture Decision Record (ADR) based on the design
review notes below. Use this exact structure:

# ADR-[number]: [Title]

## Status
[Proposed / Accepted / Superseded]

## Context
[2-3 sentences: what problem and why now]

## Decision
[What was decided, 2-3 sentences]

## Consequences
[Bulleted: positive and negative trade-offs]

## Open Questions
[Bulleted: unresolved items affecting this decision]

---
Generate ONE ADR for the core decision: migrating from monolith
to microservices. Under 300 words.

Notes:
[Paste Sam's notes here]
"""

print("Format D: ADR ready for the team's architecture docs.")

Format D: ADR ready for the team's architecture docs.


## output of prompt_adr →

# ADR: Migrate Checkout System from Monolith to Microservices

## Status

Accepted

## Context

The current checkout monolith experiences ~4s response times under peak load, significantly exceeding the target of sub-500ms. As traffic and feature complexity grow, the monolith limits scalability, independent deployment, and performance optimization. A redesign is required to meet performance goals and enable faster iteration.

## Decision

The team decided to decompose the checkout system into three independent microservices: cart-service, payment-service, and order-service. The architecture will follow an event-driven model, where the payment-service publishes order completion events consumed by the order-service, enabling decoupled workflows and scalability.

## Consequences

* Improved scalability and performance through independent service scaling
* Faster development velocity with isolated services and deployments
* Clear separation of concerns across cart, payment, and order domains
* Increased operational complexity, including service coordination and monitoring
* Need for robust inter-service communication and failure handling
* Additional infrastructure requirements (e.g., VPC for PCI compliance, message broker)
* Higher overhead in debugging and distributed tracing

## Open Questions

* Which message broker to adopt (Kafka vs RabbitMQ)?
* How to handle in-flight carts during deployments (blue-green vs canary)?
* What API rate limiting strategy and thresholds should be enforced?
* What is the rollback strategy for failed deployments?
* What are the timelines and criteria for load testing before go-live?


### The Principle of Output Formatting

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/190/808/original/1.png?1776328172)


*[Run the JSON and ADR formats live. JSON impresses engineers who haven't seen AI produce machine-readable output. The ADR impresses anyone who's struggled to write one from scratch.]*

> "Four techniques in your toolkit now: RCTFC structure, few-shot examples, chain-of-thought reasoning, and format control. How do you get good at combining them? The same way you get good at anything in software: practice and iteration."

---
---

# SECTION 5 -- GUIDED EXERCISE: REFINING A PROMPT IN FIVE ROUNDS

**Time: 20 minutes** | **Goal:** Students experience starting bad and getting good through deliberate iteration.

> "Almost nobody writes a great prompt on the first try. The best prompt engineers iterate -- the same way the best developers don't ship their first draft of code.
>
> Prompt refinement IS debugging. Write, read the output, identify the failure, fix one thing, run again.
>
> I'll take a bad prompt and refine it five times, live. Follow along."

### The Task: Sam Needs to Email His Tech Lead

Sam needs to email Maya a summary of his action items, flag concerns, and propose a plan.

In [ ]:
# ROUND 1 -- The starting point
r1 = "Write an email about the design review action items."
print("Round 1: Vague. No audience, no context, no source material.")
print("Expected: generic, useless email about hypothetical action items.")

In [ ]:
# ROUND 2 -- Add Context + source material
r2 = """
I attended a system design review for rewriting our checkout service.
Write an email to my tech lead summarising the action items.

Notes:
[Paste Sam's design review notes here]
"""
print("Round 2: Now grounded in real data. But output is likely a wall of text.")

In [ ]:
# ROUND 3 -- Add Role + Format
r3 = """
Role: You are a backend developer writing a follow-up email
to your tech lead after a design review.

Context: We held a design review for rewriting our checkout
monolith into microservices.

Task: Draft an email to Maya (Tech Lead) with these sections:
1. Subject line
2. One-line summary of the review outcome
3. My action items (what I own)
4. Items I'm concerned about (with reasons)
5. Questions for the next review

Notes:
[Paste Sam's design review notes here]
"""
print("Round 3: Structured. But might be too long, tone might be off.")

In [ ]:
# ROUND 4 -- Add Constraints
r4 = """
Role: Backend developer writing a follow-up email to tech lead.

Context: Design review for checkout monolith-to-microservices rewrite.
Internal email. Maya reads on her phone between meetings.

Task: Draft email with these sections:
1. Subject line (specific to this review)
2. One-line summary
3. My committed items (table: Task | Deadline | Status)
4. Concerns (max 3, each with downstream impact and suggested fix)
5. Open questions for 18 Apr review (bulleted, max 4)

Constraints:
- Under 300 words.
- Tone: direct, professional, peer-to-peer.
- Each concern must name the specific downstream impact.
- Sign off as Sam.

Notes:
[Paste Sam's notes here]
"""
print("Round 4: Tight, professional, practical. Genuinely useful.")

In [ ]:
# ROUND 5 -- Add chain-of-thought for the analysis section
r5 = """
Role: Backend developer writing a follow-up email to tech lead.

Context: 50-min design review for checkout monolith rewrite into
three microservices (cart, payment, order). Internal email.
Maya reads on her phone between meetings.

Task: Draft email with these sections:
1. Subject line (specific, not generic)
2. One-line summary
3. My committed items (table: Task | Deadline | Blocked By)
4. Top 3 concerns -- for each:
   a) The issue
   b) What downstream work it blocks
   c) My suggested mitigation
5. Open questions for 18 Apr review (bulleted, max 4)

Before writing Concerns, internally map every task's dependencies
to find which delays cascade the most. Select the three with the
highest blast radius.

Constraints:
- Under 350 words.
- Tone: direct, peer-to-peer.
- Do not invent facts. Use TBD for unknowns.
- Sign off as Sam.

Notes:
[Paste Sam's notes here]
"""
print("Round 5: Production-quality email with dependency-aware risk analysis.")
print("Compare this to Round 1. Same AI. Same topic.")

### What Just Happened

| Round | What Changed | RCTFC Element | Output Quality |
|-------|-------------|---------------|----------------|
| 1 | Bare question, no material | None | AI invented everything |
| 2 | Added source notes and recipient | Context | Factual but unstructured |
| 3 | Added persona and sections | Role + Format | Structured but generic |
| 4 | Added length, tone, specificity rules | Constraints | Tight and professional |
| 5 | Added chain-of-thought for analysis | Reasoning layer | Dependency-aware risk analysis |

**The debugging parallel:**

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/190/812/original/2.png?1776329421)


Same loop. Same discipline.

---
---

# SECTION 6 -- MINI BUILD: THREE PROMPT CHALLENGES FOR LEARNERS



> "Three challenges. Each tests a different combination. Write your prompt, run it, read the output, iterate at least once."

---

## Challenge 1: "The Code Reviewer"

**The setup:** Here's a Python function from a junior developer's PR:

```python
def get_user_orders(user_id, db):
    orders = db.execute(f"SELECT * FROM orders WHERE user_id = {user_id}")
    result = []
    for o in orders:
        if o['status'] != 'cancelled':
            d = {}
            d['id'] = o['id']
            d['total'] = o['total']
            d['date'] = str(o['created_at'])
            d['items'] = len(o['line_items'])
            result.append(d)
    return result
```

**The mission:** Write a prompt that produces a code review a senior engineer would actually write -- not a generic list, but one that identifies the critical issue first, explains the WHY, and provides corrected code.

**You pass if:**
- SQL injection vulnerability is identified FIRST (it's the critical issue)
- Each comment explains WHY it matters, not just what to change
- Corrected code is provided
- Tone is constructive and educational

**Techniques to use:** RCTFC + Chain-of-thought (triage by severity before commenting)

In [ ]:
# CHALLENGE 1 -- Write your prompt, run it in your AI platform
# [YOUR PROMPT HERE]

print("Does the SQL injection lead the review? Does each comment explain WHY?")
print("If not, iterate.")

---

## Challenge 2: "The Incident Commander"

**The setup:** Slack thread from a production incident:

> **#incidents -- 10 Apr 2025**
>
> [10:14] **jatin-sre:** checkout-service throwing 500s. Error rate spiked from 0.1% to 34% in 5 min.
>
> [10:16] **maya-lead:** payment-service timing out. Redis connection pool exhausted.
>
> [10:18] **sam-backend:** Logs show redis maxed at 50 connections. 200+ concurrent checkouts.
>
> [10:19] **jatin-sre:** Scaling redis pool to 200. ETA 3 min.
>
> [10:22] **maya-lead:** Error rate dropping. Down to 8%.
>
> [10:25] **jatin-sre:** Pool resize complete. Error rate 0.3%. Stable.
>
> [10:26] **pooja-pm:** How many customers affected? Getting support tickets.
>
> [10:28] **sam-backend:** ~340 failed checkouts in the 11-minute window.
>
> [10:30] **maya-lead:** Incident resolved. Sam, write the post-mortem.

**The mission:** Write a prompt that transforms this into a post-mortem.

**You pass if:**
- Has: Summary, Timeline, Root Cause (using 5-Whys), Impact, Resolution, Action Items
- Timeline uses exact timestamps
- Action items are about PREVENTION, not just resolution
- Blameless tone

**Techniques:** RCTFC + Format control + Chain-of-thought (reason about root cause depth)

In [ ]:
# CHALLENGE 2 -- Write your prompt and run it
# [YOUR PROMPT HERE]

print("Does the root cause go DEEPER than 'Redis pool was too small'?")
print("Does it ask WHY the pool was undersized in the first place?")

---

## Challenge 3: "One Meeting, Four Audiences"

**The setup:** Sam's original design review notes.

**The mission:** Write **four separate prompts** producing four outputs:

| Output | Audience | Key Requirement |
|--------|---------|----------------|
| **A. Executive summary** | VP of Engineering | Under 100 words, zero jargon, focus on dates and risks |
| **B. Sprint tickets** | Jira | Each action item as a ticket with acceptance criteria |
| **C. Dependency graph** | Engineering team | Which tasks block which, as a text-based graph or table |
| **D. API contract draft** | Frontend dev Rina | OpenAPI-style spec with endpoints, methods, request/response |

**Techniques:** RCTFC for each + Format control (completely different structure per output)

In [ ]:
# CHALLENGE 3 -- Write four separate prompts

# OUTPUT A: VP Executive Summary
# [YOUR PROMPT]

# OUTPUT B: Sprint Tickets
# [YOUR PROMPT]

# OUTPUT C: Dependency Graph
# [YOUR PROMPT]

# OUTPUT D: API Contract
# [YOUR PROMPT]

print("Same notes. Four audiences. Four completely different outputs.")
print("This is the real power of prompt engineering.")

---
---

# SECTION 7 -- WRAP-UP: THE PROMPT ENGINEER'S CHECKLIST

**Time: 10 minutes** | **Goal:** Cement techniques into a reusable checklist.

### Five takeaways:

**1. The anatomy of a great prompt (RCTFC).**
Role, Context, Task, Format, Constraints. Same principle as a well-written Jira ticket: specificity in, quality out. The AI responds to detailed specs the same way a developer does.

**2. Zero-shot vs few-shot prompting.**
When output must match a specific template, show 2-3 examples directly in the prompt. The examples ARE the instruction -- the same way showing a contractor three of your best PRs teaches your style better than a written guideline.

**3. Chain-of-thought prompting.**
For analysis, comparison, or multi-step reasoning, tell the AI to think step by step before answering. This is rubber duck debugging applied to AI -- articulating the reasoning catches errors that jumping to conclusions would miss.

**4. Output formatting control.**
Define output like an API response schema: exact columns, section headers, key names, example rows. If you can sketch the output in 10 seconds, that sketch belongs in the prompt.

**5. Iterative refinement.**
Prompting is debugging, not dictation. Write, read, diagnose, fix one thing, repeat. Nobody ships their first draft of code. Don't ship your first draft of a prompt.

### The Prompt Enginnering Toolkit

![image](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/190/814/original/3.png?1776329614)



# Techniques Reference

| Technique | What It Is | When to Use | Developer Parallel |
|-----------|-----------|------------|-------------------|
| **RCTFC** | Role + Context + Task + Format + Constraints | Every non-trivial prompt | Writing a detailed Jira ticket |
| **Zero-shot** | Instructions only, no examples | General tasks, any format OK | "Build a login page" |
| **Few-shot** | Instructions + 2-3 examples | Must match specific style/template | "Build it like these three" |
| **Chain-of-thought** | "Think step by step" | Analysis, debugging, comparison | Rubber duck debugging |
| **Format control** | Show exact output shape | Always for non-trivial output | Defining an API response schema |
| **Iterative refinement** | Write, read, fix, repeat | Every important prompt | Code review cycle |


